# 05 Embedding baselines (Phase 5, part 1)

Compares **pretrained** embedding models on the **dev** queries from notebook 04, with and without image captions:

- `text` = title, brand, type, color, bullets, description, keywords
- `text_full` = `text` + the BLIP image description

Systems compared: four pretrained embedding models, TF-IDF keyword baselines, and a hybrid (dense + TF-IDF).
Metrics: Precision@5, Precision@10, MRR, nDCG@10, with a 95% bootstrap interval, plus query latency.

**Rules:** evaluate on `dev` only while experimenting. The `test` queries are used once, at the very end (Phase 9).

**Caveat:** the weak labels come from regexes on product names and bullet points, so lexical methods (TF-IDF) get an advantage. Compare systems with each other, and look at narrow vs broad queries separately.

In [ ]:
import os, re, json, time, hashlib
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
!pip install -q sentence-transformers

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Use the same folder as notebook 04
PROJECT_DIR = '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk'
#PROJECT_DIR = '/content/drive/MyDrive/shoptalk'

In [ ]:
df = pd.read_parquet(f'{PROJECT_DIR}/data/listings_en_final.parquet')
records = json.load(open(f'{PROJECT_DIR}/data/eval/eval_queries.json'))
assert {'text', 'text_full', 'item_id', 'price', 'name_en'} <= set(df.columns)

dev = [r for r in records if r['split'] == 'dev']      # experiment on dev only
N = len(df)
item_ids = df['item_id'].to_numpy()
prices = df['price'].to_numpy()
id2idx = {i: k for k, i in enumerate(item_ids)}

rel_bool = {}
for r in records:
    b = np.zeros(N, dtype=bool)
    b[[id2idx[i] for i in r['relevant_ids']]] = True
    rel_bool[r['id']] = b

print('products:', N, '| dev queries:', len(dev), '| test queries (do not touch):', len(records) - len(dev))

## Metrics

- **P@k**: share of the top-k results that are relevant.
- **MRR**: 1 / rank of the first relevant result (looked for in the top 100).
- **nDCG@10**: rewards relevant results appearing earlier.
- Price limits (e.g. "under $30") are applied as a **filter before ranking**, the same way the final system will do it.

In [ ]:
LOG2 = 1 / np.log2(np.arange(2, 12))

def rank_top(scores, elig, k=100):
    s = np.where(elig, scores, -np.inf)
    top = np.argpartition(-s, k)[:k]
    return top[np.argsort(-s[top])]

def query_metrics(ranked, rel, n_rel):
    hits = rel[ranked]
    pos = np.flatnonzero(hits)
    return {
        'P@5': hits[:5].mean(),
        'P@10': hits[:10].mean(),
        'MRR': 1.0 / (pos[0] + 1) if len(pos) else 0.0,
        'nDCG@10': (hits[:10] * LOG2).sum() / LOG2[:min(n_rel, 10)].sum(),
    }

def elig_mask(q):
    return np.ones(N, dtype=bool) if q['max_price'] is None else prices < q['max_price']

def evaluate(score_matrix, queries):
    rows = []
    for qi, q in enumerate(queries):
        ranked = rank_top(score_matrix[qi], elig_mask(q))
        rows.append({'id': q['id'], 'query': q['query'], 'n_relevant': q['n_relevant'],
                     **query_metrics(ranked, rel_bool[q['id']], q['n_relevant'])})
    return pd.DataFrame(rows)

def bootstrap_ci(values, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(values)
    means = [rng.choice(v, len(v), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

summary_rows, per_query = [], {}

def record(system, variant, res, **extra):
    lo, hi = bootstrap_ci(res['P@5'])
    per_query[(system, variant)] = res
    summary_rows.append({'system': system, 'text': variant,
                         'P@5': res['P@5'].mean(), 'P@5 95% CI': f'{lo:.2f}-{hi:.2f}',
                         'P@10': res['P@10'].mean(), 'MRR': res['MRR'].mean(),
                         'nDCG@10': res['nDCG@10'].mean(), **extra})

## Keyword baselines (TF-IDF)

TF-IDF over product names (what you ran in notebook 04), over `text`, and over `text_full`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

dev_queries_text = [q['semantic_query'] for q in dev]
tfidf_scores = {}
for col in ['name_en', 'text', 'text_full']:
    vec = TfidfVectorizer(stop_words='english', sublinear_tf=True)
    M = vec.fit_transform(df[col])
    S = (vec.transform(dev_queries_text) @ M.T).toarray()     # rows are L2-normalized, so dot product = cosine
    tfidf_scores[col] = S
    record('TF-IDF', col, evaluate(S, dev))

print(pd.DataFrame(summary_rows).round(3).to_string(index=False))

## Pretrained embedding models

Four small, fast models. Each has its own query/passage prefix convention, which matters for BGE and E5.
All texts are truncated to 256 tokens. Embeddings are cached on Drive, so reruns are quick.
To add a model, add an entry to `MODELS`.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELS = {
    'MiniLM-L6':  dict(name='sentence-transformers/all-MiniLM-L6-v2', q_prefix='', d_prefix=''),
    'BGE-small':  dict(name='BAAI/bge-small-en-v1.5', q_prefix='Represent this sentence for searching relevant passages: ', d_prefix=''),
    'E5-small':   dict(name='intfloat/e5-small-v2', q_prefix='query: ', d_prefix='passage: '),
    'GTE-small':  dict(name='thenlper/gte-small', q_prefix='', d_prefix=''),
}
VARIANTS = {'text': df['text'].tolist(), 'text_full': df['text_full'].tolist()}

EMB_DIR = f'{PROJECT_DIR}/data/embeddings'
os.makedirs(EMB_DIR, exist_ok=True)

loaded = {}
def get_model(tag):
    if tag not in loaded:
        m = SentenceTransformer(MODELS[tag]['name'], device=device)
        m.max_seq_length = 256
        if device == 'cuda':
            m.half()
        loaded[tag] = m
    return loaded[tag]

doc_emb, enc_minutes = {}, {}
for tag, cfg in MODELS.items():
    for variant, texts in VARIANTS.items():
        path = f'{EMB_DIR}/{tag}__{variant}.npy'
        if os.path.exists(path):
            doc_emb[(tag, variant)] = np.load(path)
            continue
        t0 = time.time()
        e = get_model(tag).encode([cfg['d_prefix'] + t for t in texts], batch_size=128,
                                  normalize_embeddings=True, convert_to_numpy=True,
                                  show_progress_bar=True).astype('float32')
        enc_minutes[(tag, variant)] = (time.time() - t0) / 60
        np.save(path, e)
        doc_emb[(tag, variant)] = e
        print(tag, variant, e.shape, f'{enc_minutes[(tag, variant)]:.1f} min')

In [ ]:
# Query-time latency (one query at a time) and model size
lat, params = {}, {}
for tag, cfg in MODELS.items():
    m = get_model(tag)
    for _ in range(3):
        m.encode(['warm up'])
    ts = []
    for q in dev:
        t0 = time.perf_counter()
        m.encode([cfg['q_prefix'] + q['semantic_query']], normalize_embeddings=True)
        ts.append((time.perf_counter() - t0) * 1000)
    lat[tag] = (np.percentile(ts, 50), np.percentile(ts, 95))
    params[tag] = sum(p.numel() for p in m.parameters()) / 1e6

dense_scores = {}
for tag, cfg in MODELS.items():
    q_emb = get_model(tag).encode([cfg['q_prefix'] + t for t in dev_queries_text],
                                  normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    for variant in VARIANTS:
        S = q_emb @ doc_emb[(tag, variant)].T
        dense_scores[(tag, variant)] = S
        record(tag, variant, evaluate(S, dev),
               dim=doc_emb[(tag, variant)].shape[1], params_M=round(params[tag], 1),
               q_ms_p50=round(lat[tag][0], 1), q_ms_p95=round(lat[tag][1], 1))
print('done')

## Hybrid: dense + TF-IDF (reciprocal rank fusion)

Combines the rankings from the best dense system and TF-IDF over `text`. RRF scores each result by 1 / (60 + rank) in each list and sums them.

In [ ]:
def rrf_eval(S_a, S_b, queries, k=100, c=60):
    rows = []
    for qi, q in enumerate(queries):
        elig = elig_mask(q)
        fused = {}
        for S in (S_a, S_b):
            for rank, idx in enumerate(rank_top(S[qi], elig, k)):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (c + rank + 1)
        ranked = np.array(sorted(fused, key=fused.get, reverse=True))
        rows.append({'id': q['id'], 'query': q['query'], 'n_relevant': q['n_relevant'],
                     **query_metrics(ranked, rel_bool[q['id']], q['n_relevant'])})
    return pd.DataFrame(rows)

dense_rows = [r for r in summary_rows if r['system'] in MODELS]
best = max(dense_rows, key=lambda r: r['P@5'])
print('Best dense system by P@5:', best['system'], '/', best['text'])
hyb = rrf_eval(dense_scores[(best['system'], best['text'])], tfidf_scores['text'], dev)
record(f"Hybrid ({best['system']} + TF-IDF)", best['text'], hyb)

## Results (dev queries)

In [ ]:
results = pd.DataFrame(summary_rows)
cols = ['system', 'text', 'P@5', 'P@5 95% CI', 'P@10', 'MRR', 'nDCG@10', 'dim', 'params_M', 'q_ms_p50', 'q_ms_p95']
results = results[cols].sort_values('P@5', ascending=False)
print(f'Random baseline P@k is about {sum(r["n_relevant"] for r in dev) / len(dev) / N:.4f}')
display(results.round(3))

# Does adding the image caption help? (same model, text vs text_full)
cap = results.pivot_table(index='system', columns='text', values='P@5')
cap['caption effect (text_full - text)'] = cap['text_full'] - cap['text']
display(cap.dropna().round(3))

In [ ]:
# Narrow (fewer than 100 relevant products) vs broad queries
rows = []
for (system, variant), res in per_query.items():
    for label, part in [('narrow', res[res['n_relevant'] < 100]), ('broad', res[res['n_relevant'] >= 100])]:
        rows.append({'system': system, 'text': variant, 'group': label, 'n_queries': len(part), 'P@5': part['P@5'].mean()})
nb = pd.DataFrame(rows).pivot_table(index=['system', 'text'], columns='group', values='P@5').reindex(columns=['narrow', 'broad'])
display(nb.round(3).sort_values('narrow', ascending=False, na_position='last'))

# Per-query view for the best dense system vs TF-IDF
b = per_query[(best['system'], best['text'])][['query', 'n_relevant', 'P@5']].rename(columns={'P@5': f"{best['system']} P@5"})
t = per_query[('TF-IDF', 'text')][['query', 'P@5']].rename(columns={'P@5': 'TF-IDF P@5'})
display(b.merge(t, on='query').sort_values(f"{best['system']} P@5").round(2))

In [ ]:
import matplotlib.pyplot as plt
plot = results.copy()
plot['label'] = plot['system'] + ' / ' + plot['text']
plot = plot.sort_values('P@5')
plt.figure(figsize=(10, 0.35 * len(plot) + 1.5))
plt.barh(plot['label'], plot['P@5'], color='#4c78a8')
plt.xlabel('Mean Precision@5 on dev queries')
plt.title('Retrieval quality by system and text variant')
plt.tight_layout()
plt.show()

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
results.to_csv(f'{PROJECT_DIR}/results/embedding_baselines_dev.csv', index=False)
print('Saved results/embedding_baselines_dev.csv')

## How to read this

- A system only counts as better if its P@5 interval does not overlap the other's. With about 19 dev queries, differences of a few points are noise.
- Check the narrow vs broad table: broad queries are easy for every method.
- If TF-IDF beats the dense models on attribute queries (men's, black, leather), that is expected. The real system will handle gender, color and price with metadata filters.
- Look at the per-query table for queries where the dense model clearly beats TF-IDF (semantic queries like the coffee gift). That is the case for embeddings.
- Next (part 2): fine-tune the best model on product-derived pairs from `train` products only, then compare against these numbers.